## Installing required packages

In [8]:
!pip install pyspellchecker 
!pip install nltk


## Importing required libraries

In [9]:
from spellchecker import SpellChecker
import tkinter as tk
from tkinter import scrolledtext, messagebox
import re
import time
from collections import Counter
import nltk
nltk.download('punkt', quiet=True)


True

## creating main autocorrect engine

In [12]:
class AutocorrectEngine:
    def __init__(self):
        self.spell = SpellChecker()
        self.custom_words = set()

    def add_custom_words(self, words):
        """Add custom words to the dictionary"""
        self.custom_words.update(words)
        self.spell.word_frequency.load_words(words)

    def identify_misspelled_words(self, text):
        """Identify misspelled words in the text"""
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        misspelled = self.spell.unknown(words)
        return list(misspelled)

    def get_corrections(self, word):
        """Get correction suggestions for a misspelled word"""
        candidates = self.spell.candidates(word)
        if candidates:
            return list(candidates)[:5]  # Return top 5 suggestions
        return []
    
    def autocorrect_text(self, text, auto_correct=True):
        """Autocorrect the entire text"""
        words = text.split()
        corrected_words = []
        corrections_made = {}
        
        for word in words:
            clean_word = re.sub(r'[^\w]', '', word.lower())
            
            if clean_word and clean_word in self.spell.unknown([clean_word]):
                if auto_correct:
                    correction = self.spell.correction(clean_word)
                    if correction and correction != clean_word:
                        corrected_word = self.preserve_formatting(word, correction)
                        corrected_words.append(corrected_word)
                        corrections_made[word] = corrected_word
                    else:
                        corrected_words.append(word)
                else:
                    corrected_words.append(word)
            else:
                corrected_words.append(word)
        
        return ' '.join(corrected_words), corrections_made
        
    def preserve_formatting(self, original, correction):
        """Preserve capitalization and punctuation from original word"""
        if original.isupper():
            return correction.upper()
        elif original.istitle():
            return correction.capitalize()
        return correction

## creating function for real time spell checking

In [13]:
class RealTimeChecker:
    def __init__(self, autocorrect_engine):
        self.engine = autocorrect_engine
        self.last_check_time = 0
        self.check_delay = 0.5

    def check_word_as_typed(self, word):
        """Check a single word as it's being typed"""
        clean_word = re.sub(r'[^\w]', '', word.lower())
        if len(clean_word) > 2:
            if clean_word in self.engine.spell.unknown([clean_word]):
                suggestions = self.engine.get_corrections(clean_word)
                return {
                    'is_misspelled': True,
                    'word': word,
                    'suggestions': suggestions
                }
        return {
            'is_misspelled': False,
            'word': word,
            'suggestions': []
        }

    def analyze_text_statistics(self, text):
        """Provide statistics about the text"""
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        misspelled = self.engine.identify_misspelled_words(text)
        
        return {
            'total_words': len(words),
            'misspelled_count': len(misspelled),
            'accuracy_percentage': round((len(words) - len(misspelled)) / len(words) * 100, 2) if words else 100,
            'misspelled_words': misspelled
        }